# Validate attributes on assignment

You want to stop an invalid value from getting into an instance. The right mechanism depends on *when* the validation needs to fire: once at construction time, every time an attribute is assigned, or across several fields at once.

## One-time validation at construction: `__post_init__`

For dataclasses, `__post_init__` runs after the generated `__init__` finishes. It's the simplest place to validate inputs.

In [ ]:
from dataclasses import dataclass

@dataclass
class Rectangle:
    width: float
    height: float

    def __post_init__(self):
        if self.width <= 0 or self.height <= 0:
            raise ValueError("sides must be positive")

print(Rectangle(3, 4))

try:
    Rectangle(-1, 4)
except ValueError as e:
    print(f"{type(e).__name__}: {e}")

This guards construction but *not* subsequent assignment. Someone can still do `r.width = -1` and break the invariant. If that matters — read on.

## Ongoing validation: `@property` setters

When you want every assignment to be checked, not just the first one, use a `@property` with a setter. The real value lives under an underscored name; the public name is the property.

In [ ]:
class Rectangle:
    def __init__(self, width, height):
        self.width = width     # triggers the setter
        self.height = height   # triggers the setter

    @property
    def width(self):
        return self._width

    @width.setter
    def width(self, value):
        if value <= 0:
            raise ValueError("width must be positive")
        self._width = value

    @property
    def height(self):
        return self._height

    @height.setter
    def height(self, value):
        if value <= 0:
            raise ValueError("height must be positive")
        self._height = value

r = Rectangle(3, 4)
try:
    r.width = -1
except ValueError as e:
    print(f"{type(e).__name__}: {e}")

Two things worth noticing:

- `__init__` just assigns — the setter does the validation. You only write the check in one place.
- This is noisy when you have many fields, because each one gets its own getter/setter pair. For that case the next option is more compact.

## Cross-field invariants: `__setattr__`

If you have a lot of fields, or an invariant that spans several of them, intercept all attribute assignments via `__setattr__`.

In [ ]:
class DateRange:
    def __init__(self, start, end):
        self.start = start
        self.end = end

    def __setattr__(self, name, value):
        if name in ("start", "end"):
            other = "end" if name == "start" else "start"
            other_value = getattr(self, other, None)
            if other_value is not None:
                if name == "start" and value > other_value:
                    raise ValueError("start cannot be after end")
                if name == "end" and value < other_value:
                    raise ValueError("end cannot be before start")
        super().__setattr__(name, value)

r = DateRange(1, 5)
try:
    r.end = 0
except ValueError as e:
    print(f"{type(e).__name__}: {e}")

Notice the `super().__setattr__(name, value)` call at the end — that's what actually stores the value. Without it, your `__setattr__` wouldn't set anything at all, and you'd have an infinite loop if you tried `self.x = value` inside it instead.

`__setattr__` is powerful but reaches for almost any attribute access, which can surprise future readers. Use it when the invariant genuinely crosses fields. For single-field validation, stay with `@property`.

## Choosing

- **`__post_init__`** when "do these inputs make sense?" is enough and the class is mostly a value type you don't mutate.
- **`@property` setter** when each field validates independently and you want changes checked.
- **`__setattr__`** when the invariant spans multiple fields, or when you have many fields and the `@property` boilerplate would be worse than the `__setattr__` indirection.

A fourth option worth mentioning: `pydantic.BaseModel` (in the pydantic library) does all of this — and more — declaratively. If you're already using pydantic for parsing data, its models give you validation as a side effect. For stdlib-only code, stick with the three options above.